# <font color="#418FDE" size="6.5" uppercase>**Data Pre-processing and Split**</font>

>Last update: 20260325.
    
By the end of this Lecture, you will be able to:
- Explain the purpose of data preprocessing techniques such as scaling, calibration, and cleaning in ML for CE data. 
- Apply appropriate methods to split tabular datasets into training-testing sets for both classification and regression tasks. 
- Differentiate preprocessing requirements for tabular, image, and time-series data for ML analysis. 


## **1. Fundamentals of Preprocessing**

### **1.1. Cleaning Inconsistent Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_01_01.jpg?v=1774474880" width="250">



>* Civil data sources often use inconsistent formats.
>* Cleaning prevents misleading patterns and improves reliability.

>* Ensure features stay consistent and meaningful.
>* Separate true events from data errors.

>* Cleaning improves fairness and reliable decisions.
>* Standardized data builds trustworthy, comparable models.



In [ ]:
#@title Python Code - Cleaning Inconsistent Data

# This example cleans civil engineering sensor data.
# We inspect inconsistent missing value markers.
# Then we make one simple plot.

# Import beginner friendly data tools.
import os
import zipfile
import numpy as np

# Import table and plotting libraries.
import pandas as pd
import matplotlib.pyplot as plt

# Download the air quality dataset.
!wget -q -O ./AirQualityUCI.zip \
"https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip"

# Unzip the downloaded dataset file.
with zipfile.ZipFile("./AirQualityUCI.zip", "r") as z:
    z.extractall("./air_quality_data")

# Build the csv file path.
csv_path = os.path.join("./air_quality_data", "AirQualityUCI.csv")

# Load the semicolon separated file.
df = pd.read_csv(csv_path, sep=";", decimal=",")

# Clean column names first
df.columns = df.columns.str.strip()

# Drop fully empty columns
df = df.dropna(axis=1, how="all")
df = df.loc[:, ~df.columns.str.contains("^Unnamed", regex=True)]

# Count the special missing marker.
raw_missing_count = (df == -200).sum().sum()

# Replace -200 with proper missing values.
df = df.replace(-200, np.nan)

# Combine date and time safely.
df["Datetime"] = pd.to_datetime(
    df["Date"].astype(str).str.strip() + " " + df["Time"].astype(str).str.strip(),
    format="%d/%m/%Y %H.%M.%S",
    errors="coerce"
)

# Standardize one text style example.
df.columns = df.columns.str.strip()

# Keep only rows with valid timestamps.
df = df.dropna(subset=["Datetime"])

# Check one important sensor column.
co_missing = df["CO(GT)"].isna().sum()

# Fill missing values by time order.
df = df.sort_values("Datetime")
df["CO(GT)"] = df["CO(GT)"].interpolate(limit_direction="both")

# Create a small summary table.
rows_count = df.shape[0]
cols_count = df.shape[1]
co_after = df["CO(GT)"].isna().sum()

# Print short teaching messages.
print("Loaded rows and columns:", rows_count, cols_count)
print("Original -200 markers found:", int(raw_missing_count))
print("CO(GT) missing before cleaning:", int(co_missing))
print("CO(GT) missing after cleaning:", int(co_after))

# Show a few simple checks.
print("Duplicate timestamps:", int(df["Datetime"].duplicated().sum()))
print("CO(GT) min and max:", round(df["CO(GT)"].min(), 2), round(df["CO(GT)"].max(), 2))

# Plot a small cleaned time slice.
plot_df = df[["Datetime", "CO(GT)"]].dropna().head(200)
plt.figure(figsize=(8, 4))
plt.plot(plot_df["Datetime"], plot_df["CO(GT)"], color="teal")

# Label the single teaching plot.
plt.title("Cleaned CO(GT) Sensor Readings")
plt.xlabel("Time")
plt.ylabel("CO(GT)")
plt.xticks(rotation=30)

# Finish the plot neatly.
plt.tight_layout()
plt.show()



### **1.2. Feature Scaling Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_01_02.jpg?v=1774474925" width="250">



>* Scaling makes features use comparable ranges.
>* Prevents large-value variables dominating model learning.

>* Scaling prevents large features dominating comparisons.
>* Standardise or normalise for balanced learning.

>* Scaling improves training stability and fairness.
>* Fit scaling on training data consistently.



In [ ]:
#@title Python Code - Feature Scaling Basics

# Feature scaling helps compare engineering variables fairly.
# This example uses a concrete strength dataset.
# We compare raw and scaled input features.

# Install Excel reader if needed.
# !pip -q install xlrd

# Download the dataset file.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Import beginner friendly libraries.
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Set a seed for consistency.
np.random.seed(42)

# Load the spreadsheet dataset.
df = pd.read_excel("./concrete.xls")

# Separate inputs and target column.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Check the expected dataset shape.
if X.shape[1] != 8:
    raise ValueError("Expected eight input features.")

# Show why scales differ across features.
raw_summary = pd.DataFrame({
    "min": X.min(),
    "max": X.max(),
    "mean": X.mean()
})

# Fit the standard scaling method.
standard_scaler = StandardScaler()
X_standard = standard_scaler.fit_transform(X)

# Fit the min max scaling method.
minmax_scaler = MinMaxScaler()
X_minmax = minmax_scaler.fit_transform(X)

# Convert scaled arrays to tables.
standard_df = pd.DataFrame(X_standard, columns=X.columns)
minmax_df = pd.DataFrame(X_minmax, columns=X.columns)

# Summarize standard scaled features.
standard_summary = pd.DataFrame({
    "mean": standard_df.mean(),
    "std": standard_df.std()
})

# Summarize min max scaled features.
minmax_summary = pd.DataFrame({
    "min": minmax_df.min(),
    "max": minmax_df.max()
})

# Print a short teaching summary.
print("Dataset shape:", df.shape)
print("Input features:", X.shape[1], "Target:", y.name)
print("Raw ranges show different measurement scales.")
print(raw_summary.round(2).head(4).to_string())
print("StandardScaler centers features near zero.")
print(standard_summary.round(2).head(4).to_string())
print("MinMaxScaler maps features into zero to one.")
print(minmax_summary.round(2).head(4).to_string())
print("Scaling helps models treat variables more fairly.")



### **1.3. Dataset Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_01_03.jpg?v=1774474955" width="250">



>* Calibration aligns measurements with real conditions.
>* It reduces sensor bias and modeling errors.

>* Calibration differs from cleaning and scaling.
>* It corrects instrument bias before modeling.

>* Calibration enables consistent, trustworthy data comparisons.
>* It improves model reliability and decisions.



In [ ]:
#@title Python Code - Dataset Calibration

# This example shows simple sensor calibration.
# We compare raw and reference pollution values.
# Civil engineering data often need calibration.

# Import beginner friendly libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import simple machine learning tools.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# Set a seed for repeatability.
np.random.seed(42)

# Download the air quality dataset.
!wget -q -O ./AirQualityUCI.zip "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip"
!unzip -o -q ./AirQualityUCI.zip

# Load the main csv file.
df = pd.read_csv(
    "./AirQualityUCI.csv",
    sep=";",
    decimal=",",
    na_values=-200
)

# Remove empty unnamed columns.
df = df.loc[:, ~df.columns.str.contains("Unnamed")]

# Keep useful calibration columns.
cols = ["PT08.S1(CO)", "CO(GT)"]
df = df[cols].dropna().copy()

# Rename columns for clarity.
df.columns = ["sensor_raw", "co_reference"]

# Keep only realistic positive values.
df = df[(df["sensor_raw"] > 0) & (df["co_reference"] > 0)].copy()

# Use a manageable sample size.
if len(df) > 800:
    df = df.sample(n=800, random_state=42)

# Prepare inputs and targets.
X = df[["sensor_raw"]]
y = df["co_reference"]

# Split data into train and test parts.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Fit a simple calibration model.
model = LinearRegression()
model.fit(X_train, y_train)

# Predict calibrated values on test data.
y_pred = model.predict(X_test)

# Build a simple baseline prediction.
baseline_value = y_train.mean()
y_base = np.full(shape=len(y_test), fill_value=baseline_value)

# Measure calibration improvement.
mae_before = mean_absolute_error(y_test, y_base)
mae_after = mean_absolute_error(y_test, y_pred)
r2_after = r2_score(y_test, y_pred)

# Print short teaching messages.
print("Rows used:", len(df))
print("Raw sensor feature: PT08.S1(CO)")
print("Reference target: CO(GT)")
print("Calibration slope:", round(model.coef_[0], 4))
print("Calibration intercept:", round(model.intercept_, 4))
print("Baseline MAE:", round(mae_before, 3))
print("Calibrated MAE:", round(mae_after, 3))
print("Test R^2:", round(r2_after, 3))

# Create one clear calibration plot.
plt.figure(figsize=(8, 6))
plt.scatter(
    X_test["sensor_raw"], y_test,
    alpha=0.5, label="Reference measurements"
)

# Sort values for a neat line.
order = np.argsort(X_test["sensor_raw"].values)
x_sorted = X_test["sensor_raw"].values[order]
y_sorted = y_pred[order]

# Draw the learned calibration line.
plt.plot(
    x_sorted, y_sorted,
    color="red", linewidth=2,
    label="Linear calibration"
)

# Add labels and teaching title.
plt.xlabel("Raw sensor reading: PT08.S1(CO)")
plt.ylabel("Reference CO concentration: CO(GT)")
plt.title("Dataset Calibration: Raw Sensor to Reference CO")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## **2. Train Test Split**

### **2.1. Why Split Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_02_01.jpg?v=1774474991" width="250">



>* Splits test prediction on unseen data.
>* They prevent misleading accuracy in engineering.

>* Splitting data reveals overfitting in tabular models.
>* Untouched test sets check true generalization.

>* Test sets enable fair model comparison.
>* They promote robust, trustworthy real-world performance.



In [ ]:
#@title Python Code - Why Split Data

# We split data for honest testing.
# This example uses concrete strength data.
# Watch training and testing scores differ.

# Install Excel reader if needed.
# !pip -q install xlrd

# Download the dataset quietly.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
import random
import numpy as np
import pandas as pd

# Import simple tools for splitting.
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

# Set seeds for repeatable results.
random.seed(42)
np.random.seed(42)
file_path = "./concrete.xls"
df = pd.read_excel(file_path)

# Check the dataset size first.
rows, cols = df.shape
print("Dataset shape:", rows, "rows and", cols, "columns")
print("Target column:", df.columns[-1])

# Separate inputs and target values.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
print("Feature count:", X.shape[1])
print("Target examples:", len(y))

# Split into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Show the split sizes clearly.
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

# Fit a flexible model deliberately.
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# Compare training and testing performance.
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
print("Training R^2:", round(train_r2, 3))
print("Testing R^2:", round(test_r2, 3))

# Interpret the score gap simply.
gap = train_r2 - test_r2
if gap > 0.2:
    print("Large gap: likely overfitting.")
else:
    print("Small gap: generalization looks better.")

print("Test data estimates future performance better.")



### **2.2. Choosing Test Ratio**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_02_02.jpg?v=1774475021" width="250">



>* Balance training needs with reliable testing.
>* Use 80/20 as a flexible starting point.

>* Large datasets need smaller test proportions.
>* Small datasets require careful reliability tradeoffs.

>* Classification needs enough examples from each class.
>* Regression tests should span realistic value ranges.



In [ ]:
#@title Python Code - Choosing Test Ratio

# This example studies train test ratios.
# We use concrete strength regression data.
# Results show evaluation changes slightly.

# Install Excel reader if needed.
# !pip install xlrd

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

# Set seeds for repeatable splitting.
np.random.seed(42)

# Download the concrete dataset file.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Load the spreadsheet into pandas.
df = pd.read_excel("./concrete.xls")

# Check the dataset loaded correctly.
if df.shape[0] < 100:
    raise ValueError("Dataset seems too small.")

# Separate inputs and target column.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Confirm matching sample counts.
if len(X) != len(y):
    raise ValueError("Input and target mismatch.")

# Choose several test ratios.
test_sizes = [0.2, 0.25, 0.3]
results = []

# Try each split ratio once.
for test_size in test_sizes:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    # Train the same regression model.
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict concrete strength values.
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)

    # Measure fit on test data.
    r2 = r2_score(y_test, y_pred)
    results.append([test_size, len(X_train), len(X_test), mae, r2])

# Put results into a small table.
results_df = pd.DataFrame(
    results,
    columns=["Test ratio", "Train rows", "Test rows", "MAE", "R2"]
)

# Round values for easier reading.
results_df["MAE"] = results_df["MAE"].round(2)
results_df["R2"] = results_df["R2"].round(3)

# Print a short teaching summary.
print("Concrete dataset shape:", df.shape)
print("Same model, different test ratios:")
print(results_df.to_string(index=False))
print("Larger test sets give stronger evaluation")
print("but leave fewer rows for training.")



### **2.3. Classification Regression Splits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_02_03.jpg?v=1774475053" width="250">



>* Classification splits should preserve class proportions.
>* Use stratified splitting, especially for rare classes.

>* Regression splits need balanced value coverage.
>* Binning helps check low-to-high target balance.

>* Match splitting method to prediction task.
>* Keep related records and rare cases balanced.



In [ ]:
#@title Python Code - Classification Regression Splits

# This example compares two splitting strategies.
# We use concrete strength engineering data.
# The goal is beginner friendly practice.

# Install xlrd if Colab needs it.
# !pip install xlrd

import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Set seeds for repeatable results.
random.seed(42)
np.random.seed(42)

# Download the concrete dataset file.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Load the spreadsheet into pandas.
df = pd.read_excel("./concrete.xls")

# Rename columns for easier reading.
df.columns = [
    "cement", "slag", "fly_ash", "water",
    "superplasticizer", "coarse_agg", "fine_agg", "age",
    "strength"
]

# Separate inputs and regression target.
X = df.drop(columns=["strength"])
y_reg = df["strength"]

# Create a simple binary class.
median_strength = y_reg.median()
y_clf = (y_reg >= median_strength).astype(int)

# Split data for regression task.
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# Split data for classification task.
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Print a few useful dataset facts.
print("Rows and columns:", df.shape)
print("Median strength MPa:", round(median_strength, 2))
print("Regression train and test:", X_train_reg.shape, X_test_reg.shape)
print("Regression target ranges:",
      round(y_train_reg.min(), 1), round(y_train_reg.max(), 1),
      round(y_test_reg.min(), 1), round(y_test_reg.max(), 1))

# Print class balance before splitting.
print("Original class counts:", y_clf.value_counts().sort_index().to_dict())
print("Train class counts:", y_train_clf.value_counts().sort_index().to_dict())
print("Test class counts:", y_test_clf.value_counts().sort_index().to_dict())

# Summarize the main lesson.
print("Regression uses random splitting for continuous targets.")
print("Classification uses stratified splitting for balanced classes.")



## **3. Preprocessing Across Data Types**

### **3.1. Tabular Data Preprocessing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_03_01.jpg?v=1774475088" width="250">



>* Check variable meaning, quality, and usefulness.
>* Convert inconsistent data into machine-readable inputs.

>* Handle mixed columns with suitable preprocessing.
>* Use domain knowledge for missing values, outliers.

>* Tabular preprocessing emphasizes consistent, meaningful columns.
>* Avoid leakage; engineer practical, reliable features.



In [ ]:
#@title Python Code - Tabular Data Preprocessing

# This example shows tabular preprocessing steps.
# We use concrete strength engineering data.
# The script inspects, checks, and scales.

# Install Excel reader for pandas.
# !pip -q install xlrd

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Set a seed for reproducibility.
np.random.seed(42)

# Download the concrete dataset file.
!wget -q -O ./concrete.xls "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"

# Load the spreadsheet into pandas.
df = pd.read_excel("./concrete.xls")

# Clean column names for readability.
df.columns = df.columns.str.strip()

# Show dataset size and columns.
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))

# Check for missing values.
missing_counts = df.isna().sum()
print("Missing values total:", int(missing_counts.sum()))

# Separate features and target column.
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Confirm feature and target shapes.
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

# Scale numerical feature columns.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert scaled data back to dataframe.
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Show one feature mean after scaling.
print("Scaled cement mean:", round(X_scaled_df.iloc[:, 0].mean(), 3))

# Show one feature standard deviation.
print("Scaled cement std:", round(X_scaled_df.iloc[:, 0].std(ddof=0), 3))

# Preview the target variable name.
print("Target column:", y.name)



### **3.2. Image Data Preprocessing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_03_02.jpg?v=1774475118" width="250">



>* Preserve spatial patterns while standardizing images.
>* Normalize, crop, and enhance key features.

>* Handle realistic image variability with careful augmentation.
>* Reduce noise while preserving visual meaning.

>* Keep annotations aligned with image changes.
>* Balance detail, realism, and computing cost.



In [ ]:
#@title Python Code - Image Data Preprocessing

# This script shows image preprocessing basics.
# Landsat pixels support land cover analysis.
# Civil engineers use geospatial image features.

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Download the training image table.
get_ipython().system('wget -q -O ./sat.trn https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.trn')
# Download the testing image table.
get_ipython().system('wget -q -O ./sat.tst https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/sat.tst')

# Set a seed for consistency.
np.random.seed(7)
# Load whitespace separated training data.
train_df = pd.read_csv('./sat.trn', sep='\\s+', header=None)
# Load whitespace separated testing data.
test_df = pd.read_csv('./sat.tst', sep='\\s+', header=None)

# Separate pixel features and labels.
X_train = train_df.iloc[:, :-1].copy()
y_train = train_df.iloc[:, -1].copy()
# Separate testing features and labels.
X_test = test_df.iloc[:, :-1].copy()
y_test = test_df.iloc[:, -1].copy()

# Check that feature counts match.
assert X_train.shape[1] == X_test.shape[1]
# Scale pixel based features only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Summarize this multispectral image dataset.
print('Dataset: Statlog Landsat Satellite image neighborhoods')
print('Use: land-cover analysis for geospatial civil engineering')
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Classes in train labels:', int(y_train.nunique()))

# Show preprocessing effects on pixel features.
print('Original first feature mean:', round(float(X_train.iloc[:, 0].mean()), 2))
print('Scaled first feature mean:', round(float(X_train_scaled[:, 0].mean()), 2))
print('Scaled first feature std:', round(float(X_train_scaled[:, 0].std()), 2))

# Confirm labels stay unchanged.
print('First five labels:', y_train.iloc[:5].tolist())
print('Scaled data ready for image based ML tasks.')



### **3.3. Time Series Preprocessing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_04/Lecture_A/image_03_03.jpg?v=1774475150" width="250">



>* Preserve time order and timestamp consistency.
>* Resample irregular sensor data before modelling.

>* Handle gaps and noise with context.
>* Create features revealing temporal patterns.

>* Split chronologically to prevent future data leakage.
>* Fit scaling and windows using past data.



In [ ]:
#@title Python Code - Time Series Preprocessing

# Time series order matters during preprocessing.
# Civil sensors often produce irregular measurements.
# Lag features help models use recent history.

import os
import random
import zipfile
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt

# Set seeds for repeatable results.
random.seed(7)
np.random.seed(7)

# Download the air quality dataset.
!wget -q -O ./AirQualityUCI.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip

# Extract the downloaded zip file.
with zipfile.ZipFile("./AirQualityUCI.zip", "r") as zip_ref:
    zip_ref.extractall("./air_quality_data")

# Build the csv file path.
file_path = os.path.join(
    "./air_quality_data", "AirQualityUCI.csv"
)

# Read the semicolon separated file.
df = pd.read_csv(
    file_path, sep=";", decimal=","
)

# Remove fully empty extra columns.
df = df.dropna(axis=1, how="all")

# Combine date and time columns.
df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H.%M.%S",
    errors="coerce"
)

# Remove rows with invalid timestamps.
df = df.dropna(subset=["Datetime"])

# Sort records by chronological order.
df = df.sort_values("Datetime")
df = df.drop_duplicates(subset=["Datetime"])
df = df.reset_index(drop=True)

# Replace dataset missing value marker.
df = df.replace(-200, np.nan)

# Keep one useful sensor variable.
df = df[["Datetime", "CO(GT)"]].copy()

# Rename the target for clarity.
df = df.rename(columns={"CO(GT)": "co"})

# Fill short gaps using time interpolation.
df["co"] = df["co"].interpolate(method="linear")

# Fill any remaining edge gaps.
df["co"] = df["co"].ffill()
df["co"] = df["co"].bfill()

# Create simple lagged predictor columns.
df["co_lag_1"] = df["co"].shift(1)
df["co_lag_3"] = df["co"].shift(3)

# Create a next step prediction target.
df["co_next_hour"] = df["co"].shift(-1)

# Remove rows made incomplete by shifting.
df = df.dropna().reset_index(drop=True)

# Split without shuffling future data.
split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

# Print a short preprocessing summary.
print("Rows after cleaning:", len(df))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("First timestamp:", df["Datetime"].min())
print("Last timestamp:", df["Datetime"].max())
print("Example columns:", list(df.columns))

# Plot the cleaned time series.
plt.figure(figsize=(10, 4))
plt.plot(df["Datetime"].iloc[:200], df["co"].iloc[:200])
plt.title("Cleaned CO time series, first 200 points")
plt.xlabel("Time")
plt.ylabel("CO")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Data Pre-processing and Split**</font>


In this lecture, you learned to:
- Explain the purpose of data preprocessing techniques such as scaling, calibration, and cleaning in ML for CE data. 
- Apply appropriate methods to split tabular datasets into training-testing sets for both classification and regression tasks. 
- Differentiate preprocessing requirements for tabular, image, and time-series data for ML analysis. 

In the next Lecture (Lecture B), we will go over 'Popular Scikit-Learn ML Models'